In [4]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion

# load R package
ro.r('.libPaths(c("/dcs/23/u2200504/R/x86_64-redhat-linux-gnu-library/4.5", .libPaths()))')
ro.r('library(bnlearn)')

In [5]:
#set up for experimentation
import os
from pathlib import Path
#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"
#test it works
cp_root

#path to put results
output_dir = project_root/"results"/"graphs_HC"
output_dir.mkdir(parents=True,exist_ok=True)

In [15]:
#run hc function
def run_hc(df, seed=1):
    #remove NA rows
    df_clean = df.dropna().copy()
    with (ro.default_converter+pandas2ri.converter).context():
        ro.globalenv["df"]=conversion.py2rpy(df_clean)
        ro.globalenv["seed"]= seed

        ro.r('''
        library(bnlearn)
        data_df <- as.data.frame(df)
        
        any_disc <- FALSE
        any_cont <- FALSE

        for (i in seq_along(data_df)) {
          col <- data_df[[i]]
          if (is.numeric(col)) {
            vals <- unique(col[!is.na(col)])
            if (length(vals) <= 6 && all(abs(vals - round(vals)) < 1e-8)) {
              data_df[[i]] <- factor(col)
              any_disc <- TRUE
            } else {
              data_df[[i]] <- as.numeric(col)
              any_cont <- TRUE
            }
          } else {
            data_df[[i]] <- factor(col)
            any_disc <- TRUE
          }
        }

        #detect if there are mixed types, continuous only, or discrete only
        if (any_disc && any_cont) {
          chosen_score <- "bic-cg"    # mixed conditional Gaussian [web:148][web:154]
        } else if (any_disc && !any_cont) {
          chosen_score <- "bde"       # discrete-only
        } else if (!any_disc && any_cont) {
          chosen_score <- "bic-g"     # Gaussian-only
        } else {
          stop("No usable variables (neither discrete nor continuous).")
        }
        
        set.seed(seed)

        #run hill climbing with conditional score
        dag_hc <- hc(data_df, score=chosen_score)

        adj <- amat(dag_hc)
        nodes <- colnames(adj)
        ''')
        
        #convert adjacency matrix in R back to python
        adj = conversion.rpy2py(ro.r("adj"))
        nodes=list(ro.r("nodes"))

    #coerce adjacency matrix into numeric 2D array
    adj=np.asarray(adj, dtype=int)
    return adj,nodes

In [7]:
import networkx as nx
import matplotlib.pyplot as plt

#graph drawing functions
def draw_graph(adj, nodes, out_path):
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i,j]==1:
                G.add_edge(src,tgt)

    plt.figure(figsize=(10,8))
    pos = nx.spring_layout(G, k=1.2, iterations=500, seed=0)
    nx.draw(G, pos, with_labels=True, labels={node: node for node in nodes},
            node_size=900, font_size=8, arrowsize=10)
    plt.savefig(out_path, dpi=150)
    plt.close()

In [16]:
#loop over all causal pitfall data and run hill climbing on each

csv_files = sorted(cp_root.rglob("*.csv"))

for csv_path in csv_files:
    #relative path
    rel = csv_path.relative_to(cp_root)
    print(f"Processing: {rel}")

    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("  Skipped (empty file)")
            continue

        # Run HC on this dataset
        adj,nodes = run_hc(df, seed=1)

        #Output schema scenario__file__HC.png
        parts = rel.parts           
        scenario = parts[0] if len(parts) > 1 else "root"
        name_no_ext = csv_path.stem

        out_name = f"{scenario}__{name_no_ext}__HC.png"
        out_path = output_dir / out_name

        # Save PNG
        draw_graph(adj, nodes, out_path)
        print(f"  Saved graph to {out_path.relative_to(project_root)}")

    except Exception as e:
        print(f"  ERROR on {rel}: {e}")

Processing: berkson_paradox/.ipynb_checkpoints/admission_bias-checkpoint.csv
  Saved graph to results/graphs_HC/berkson_paradox__admission_bias-checkpoint__HC.png
Processing: berkson_paradox/.ipynb_checkpoints/loan_approval_bias-checkpoint.csv
  Saved graph to results/graphs_HC/berkson_paradox__loan_approval_bias-checkpoint__HC.png
Processing: berkson_paradox/.ipynb_checkpoints/movie_success_bias-checkpoint.csv
  Saved graph to results/graphs_HC/berkson_paradox__movie_success_bias-checkpoint__HC.png
Processing: berkson_paradox/admission_bias.csv
  Saved graph to results/graphs_HC/berkson_paradox__admission_bias__HC.png
Processing: berkson_paradox/hiring_bias.csv
  Saved graph to results/graphs_HC/berkson_paradox__hiring_bias__HC.png
Processing: berkson_paradox/hospital_data.csv
  Saved graph to results/graphs_HC/berkson_paradox__hospital_data__HC.png
Processing: berkson_paradox/loan_approval_bias.csv
  Saved graph to results/graphs_HC/berkson_paradox__loan_approval_bias__HC.png
Process